# M05 · Demostración computacional: espectro, SVD y rango bajo

Este notebook acompaña `lesson.md`. La regla de trabajo es deliberada: **primero se formula una afirmación matemática; después NumPy la comprueba**. Los resultados numéricos no sustituyen la explicación.



In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)


def relative_frobenius_error(original, approximation):
    return np.linalg.norm(original - approximation, "fro") / np.linalg.norm(original, "fro")


def reconstruct(U, s, Vt, k):
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]


def sign_invariant_alignment(a, b):
    """Cosine alignment ignoring the unavoidable global sign ambiguity."""
    return abs(float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))))


X = np.array([
    [4., 5., 0., 0.],
    [3., 4., 0., 0.],
    [0., 1., 5., 4.],
    [0., 1., 4., 3.],
    [5., 6., 0., 0.],
])
print("X shape:", X.shape)



X shape: (5, 4)


## 01 · Pares propios en una matriz simétrica

Para una matriz cuadrada, un vector no nulo $v$ es propio si $Av=\lambda v$. En una matriz real simétrica podemos elegir una base ortonormal de autovectores.



In [2]:
A = np.array([[2., 1.], [1., 2.]])
eigenvalues, Q = np.linalg.eigh(A)
residuals = [np.linalg.norm(A @ Q[:, i] - eigenvalues[i] * Q[:, i]) for i in range(2)]
print("autovalores:", eigenvalues)
print("Q^T Q:\n", Q.T @ Q)
print("residuos ||Av-lambda v||:", residuals)
assert np.allclose(Q.T @ Q, np.eye(2))
assert max(residuals) < 1e-12



autovalores: [1. 3.]
Q^T Q:
 [[1. 0.]
 [0. 1.]]
residuos ||Av-lambda v||: [np.float64(0.0), np.float64(0.0)]


## 02 · `eigh` frente a `eig`

`eigh` explota la simetría: devuelve autovalores reales ordenados y autovectores ortonormales. `eig` es la rutina general. Que ambas coincidan aquí no las vuelve intercambiables para toda matriz.



In [ ]:
w_general, V_general = np.linalg.eig(A)
print("eig:", np.sort(w_general))
print("eigh:", eigenvalues)
assert np.allclose(np.sort(w_general), eigenvalues)



## 03 · El puente rectangular correcto

Una matriz $5\times4$ no tiene autovalores propios porque no representa una transformación de un espacio en sí mismo. En cambio, $X^TX$ y $XX^T$ son cuadradas, simétricas y semidefinidas positivas.



In [ ]:
gram_right = X.T @ X
gram_left = X @ X.T
lambda_right = np.linalg.eigvalsh(gram_right)
lambda_left = np.linalg.eigvalsh(gram_left)
print("X^T X:", gram_right.shape, "autovalores:", lambda_right)
print("X X^T:", gram_left.shape, "autovalores:", lambda_left)
assert np.min(lambda_right) > -1e-10
assert np.min(lambda_left) > -1e-10



## 04 · Dimensiones de la SVD reducida

Con $X\in\mathbb{R}^{m\times n}$ y $r=\min(m,n)$, NumPy devuelve $U\in\mathbb{R}^{m\times r}$, $s\in\mathbb{R}^{r}$ y $V^T\in\mathbb{R}^{r\times n}$ cuando `full_matrices=False`.



In [ ]:
U, s, Vt = np.linalg.svd(X, full_matrices=False)
print("U, s, Vt:", U.shape, s.shape, Vt.shape)
print("singulares:", s)
assert U.shape == (5, 4) and s.shape == (4,) and Vt.shape == (4, 4)
assert np.allclose(U @ np.diag(s) @ Vt, X)



## 05 · Lectura geométrica: $V^T\rightarrow\Sigma\rightarrow U$

$V^T$ expresa una entrada en coordenadas singulares; $\Sigma$ escala cada dirección; $U$ reconstruye la salida. La igualdad se verifica con un vector concreto, no solo por inspección.



In [ ]:
x = np.array([1., -1., 2., 0.5])
coordinates = Vt @ x
scaled = s * coordinates
reconstructed_output = U @ scaled
print("coordenadas V^T x:", coordinates)
print("salida U Sigma V^T x:", reconstructed_output)
assert np.allclose(reconstructed_output, X @ x)



## 06 · Valores singulares y espectro no son sinónimos

Se cumple $\sigma_i^2=\lambda_i(X^TX)$, ordenando ambos conjuntos de forma compatible. Pero los autovalores de una matriz cuadrada general pueden diferir radicalmente de sus valores singulares.



In [ ]:
positive_lambda = np.linalg.eigvalsh(X.T @ X)[::-1]
B = np.array([[0., 1.], [0., 0.]])
print("sigma(X)^2:", s**2)
print("lambda(X^T X):", positive_lambda)
print("lambda(B):", np.linalg.eigvals(B), "sigma(B):", np.linalg.svd(B, compute_uv=False))
assert np.allclose(s**2, positive_lambda)
assert not np.allclose(np.sort(np.abs(np.linalg.eigvals(B))), np.sort(np.linalg.svd(B, compute_uv=False)))



## 07 · Aproximaciones de rango bajo

$X_k=U_k\Sigma_kV_k^T$ tiene rango a lo sumo $k$. No se eliminan columnas arbitrariamente: se conservan los términos singulares de mayor escala.



In [ ]:
approximations = {k: reconstruct(U, s, Vt, k) for k in range(1, 5)}
for k, Xk in approximations.items():
    print(f"k={k}: rango={np.linalg.matrix_rank(Xk)}, error_rel_F={relative_frobenius_error(X, Xk):.5f}")
    assert np.linalg.matrix_rank(Xk, tol=1e-10) <= k



## 08 · Error de reconstrucción y elección de $k$

La norma de Frobenius acumula toda la pérdida; la norma espectral identifica la peor dirección residual. La selección de $k$ requiere un criterio declarado.



In [ ]:
rows = []
for k, Xk in approximations.items():
    rel_f = relative_frobenius_error(X, Xk)
    retained = np.sum(s[:k] ** 2) / np.sum(s ** 2)
    spectral = np.linalg.norm(X - Xk, 2)
    rows.append((k, rel_f, retained, spectral))
print(" k | error_rel_F | energia_algebraica | error_2")
for row in rows:
    print(f" {row[0]} | {row[1]:.5f}      | {row[2]:.5f}            | {row[3]:.5f}")
assert np.isclose(rows[1][1], 0.01941, atol=5e-5)
assert np.isclose(rows[1][2], 0.99962, atol=5e-5)



## 09 · Lectura del espacio latente sin exceso semántico

Las filas de $V^T$ describen combinaciones de características y $U\Sigma$ da coordenadas de observaciones. Los signos globales son arbitrarios: la interpretación debe ser invariante a cambiar simultáneamente el signo de un par singular.



In [ ]:
scores = U[:, :2] * s[:2]
print("dos primeras direcciones de V^T:\n", Vt[:2])
print("coordenadas latentes U_2 Sigma_2:\n", scores)
print("pesos absolutos por bloque:", np.abs(Vt[:2, :2]).sum(axis=1), np.abs(Vt[:2, 2:]).sum(axis=1))
assert np.allclose((-U[:, 0]) * s[0], -scores[:, 0])



## 10 · Puente correcto hacia PCA

PCA exige centrar las variables. La SVD de $X_c=X-\bar X$ entrega las direcciones principales en $V$; la varianza muestral por componente es $s_i^2/(m-1)$.



In [ ]:
X_centered = X - X.mean(axis=0, keepdims=True)
Uc, sc, Vtc = np.linalg.svd(X_centered, full_matrices=False)
covariance = X_centered.T @ X_centered / (X.shape[0] - 1)
pca_values, pca_vectors = np.linalg.eigh(covariance)
order = np.argsort(pca_values)[::-1]
print("varianza desde SVD:", sc**2 / (X.shape[0] - 1))
print("autovalores covarianza:", pca_values[order])
assert np.allclose(sc**2 / (X.shape[0] - 1), pca_values[order])
assert all(sign_invariant_alignment(Vtc[i], pca_vectors[:, order[i]]) > 1 - 1e-10 for i in range(4))



## 11 · Flujo NumPy auditable

Antes de ejecutar: declarar forma, rutina, norma y tolerancia. Después: comprobar reconstrucción, ortogonalidad y monotonía de los valores singulares.



In [ ]:
reconstruction_residual = np.linalg.norm(X - U @ np.diag(s) @ Vt, "fro")
orthogonality_u = np.linalg.norm(U.T @ U - np.eye(4), "fro")
orthogonality_v = np.linalg.norm(Vt @ Vt.T - np.eye(4), "fro")
print("residuo reconstrucción:", reconstruction_residual)
print("residuos ortogonalidad U/V:", orthogonality_u, orthogonality_v)
assert reconstruction_residual < 1e-12
assert orthogonality_u < 1e-12 and orthogonality_v < 1e-12
assert np.all(np.diff(s) <= 0)



## 12 · Rango algebraico y rango numérico

`matrix_rank` usa una tolerancia. Por eso “el rango” computacional depende de escala y criterio. Para esta matriz, el rango exacto es 4, pero un criterio relativo del 3 % de $\sigma_1$ produce rango efectivo 2.



In [ ]:
default_rank = np.linalg.matrix_rank(X)
relative_tolerance = 0.03 * s[0]
effective_rank = int(np.sum(s > relative_tolerance))
print("rango con tolerancia por defecto:", default_rank)
print("tolerancia relativa elegida:", relative_tolerance)
print("rango efectivo declarado:", effective_rank)
assert default_rank == 4 and effective_rank == 2



## Cierre

El cálculo respalda cuatro afirmaciones: el puente espectral usa $X^TX$ y $XX^T$; la SVD reconstruye exactamente; $k=2$ es defendible bajo un error relativo de Frobenius cercano a 1.94 %; y una dirección latente no adquiere significado semántico por sí sola.
